<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong>
        Importando bibliotecas
    </strong>
</p>

In [52]:
import pandas as pd

In [53]:
def ler_arquivo(nome_arquivo):
    try:
        return pd.read_excel(f'bases/{nome_arquivo}.xlsx')
    except FileNotFoundError as e:
        return f'Erro: {e}'

In [54]:
df = ler_arquivo('dados de entregas')

SLA

In [55]:
df

,id_entrega,filial,cliente,status,descricao_entrega,tipo_servico,valor_frete,data_postagem,prazo_previsto,data_prevista,data_entrega,dias_atraso
0,3619,SC,Americanas,ENTREGUE,Extravio temporário,ECONÔMICO,25.16,2026-01-09 20:43:35,5,2026-01-14 20:43:35,2026-01-15 20:43:35,1
1,4850,RJ,Amazon,ENTREGUE,Entrega reagendada,NORMAL,38.17,2026-01-06 20:43:35,5,2026-01-11 20:43:35,2026-01-10 20:43:35,0
2,3232,MG,Magazine Luiza,EM TRÂNSITO,Cliente ausente no local,NORMAL,311.62,2026-02-03 20:43:35,1,2026-02-04 20:43:35,2026-02-05 20:43:35,1
3,3944,RS,Amazon,EM TRÂNSITO,Cliente ausente no local,NORMAL,65.46,2026-01-17 20:43:35,7,2026-01-24 20:43:35,2026-01-26 20:43:35,2
4,2633,SP,Amazon,EM TRÂNSITO,Problema operacional na rota,ECONÔMICO,104.62,2026-02-03 20:43:35,2,2026-02-05 20:43:35,2026-02-05 20:43:35,0
...,...,...,...,...,...,...,...,...,...,...,...,...
5195,3997,SC,Shopee,ENTREGUE,Entrega normal,NORMAL,38.12,2026-01-23 20:43:35,7,2026-01-30 20:43:35,2026-01-31 20:43:35,1
5196,3250,PR,Mercado Livre,ENTREGUE,Problema operacional na rota,ECONÔMICO,69.88,2026-01-19 20:43:35,2,2026-01-21 20:43:35,2026-01-19 20:43:35,0
5197,2834,PR,Magazine Luiza,EM TRÂNSITO,Extravio temporário,ECONÔMICO,152.43,2026-01-15 20:43:35,7,2026-01-22 20:43:35,2026-01-24 20:43:35,2
5198,507,MG,Amazon,EM TRÂNSITO,Entrega EXPRESSA,ECONÔMICO,27.43,2026-02-03 20:43:35,2,2026-02-05 20:43:35,2026-02-07 20:43:35,2


In [56]:
# Total de entregas sem atraso
#shape [0] retornar a quantidade de entregas
sla_ok = df.loc[
    (df['dias_atraso'] == 0) & (df['status'] == 'ENTREGUE')
].shape[0]

# Total de entregas 
df_total_entregues = df.loc[
    df['status'] == 'ENTREGUE'
]

#Percentual SLA
sla_percentual = (sla_ok / df_total_entregues.shape[0]) * 100

round(sla_percentual,1)

37.2

KPI - Entregas atrasadas

SLA - Identificando onde as operações funcionam melhor

In [57]:
sla_filial = df_total_entregas.groupby('filial')['dias_atraso'].mean()
round(sla_filial,2)


NameError: name 'df_total_entregas' is not defined

In [ ]:

sla_filial = df_total_entregues.groupby('filial')['dias_atraso'].apply(
    lambda grupo: (grupo == 0).mean() * 100
).sort_values(ascending=False) #ordena do maior para o menor (Decrescente)

round(sla_filial,2).reset_index(name='% entregas no prazo')


,filial,% entregas no prazo
0,RJ,42.08
1,PR,38.34
2,MG,38.25
3,SP,35.76
4,RS,34.93
5,SC,34.76


In [ ]:
#validando palavra chave sem trabalhar com dataframe (tabela)

desc = 'python é bom demais, a linguagem mais traquila que existe'

palavras = desc.split()

try:
    palavras.index('Python')
except ValueError as e:
    print(e)

list.index(x): x not in list


Padronização da descrição

In [ ]:
df['descricao_entrega'] = (
    df['descricao_entrega']
    .str.lower() #minusculo
    .str.strip() #se tiver espaço no inicio ou final, remove espaços
)

Separar as entregas atrasadas

In [ ]:
df_atrasos = df.loc[ df['dias_atraso'] > 0].copy()

Categorizando as causas dos atrasos

In [58]:
df_atrasos.loc[
    df_atrasos['descricao_entrega'].str.contains('chuva', na=False),
    'causa_atraso'
    ] = "Clima"

df_atrasos.loc[
    df_atrasos['descricao_entrega'].str.contains('ausente', na=False),
    'causa_atraso'
    ] = "Ausente"


df_atrasos.loc[
    df_atrasos['descricao_entrega'].str.contains('extravio|avaria', na=False),
    'causa_atraso'
    ] = "Perdas"

df_atrasos['causa_atraso'] = df_atrasos['causa_atraso'].fillna("Outros")

df_atrasos['causa_atraso'].head(10)

0      Perdas
2     Ausente
3     Ausente
5       Clima
6      Outros
7      Outros
9      Perdas
12     Outros
13     Outros
14     Outros
Name: causa_atraso, dtype: str

Principais causas de atrasos

In [59]:
round(df_atrasos['causa_atraso'].value_counts(normalize =True) *100, 1).reset_index(name='%')

,causa_atraso,%
0,Outros,61.3
1,Perdas,13.4
2,Ausente,12.7
3,Clima,12.7


Atrados por filial

In [67]:
causas_por_filiais = df_atrasos \
    .groupby(['filial','causa_atraso']) \
    .size() \
    .reset_index(name='Qtd') \
    .sort_values(['filial','Qtd'], ascending=[True,False])

causas_por_filiais

,filial,causa_atraso,Qtd
2,MG,Outros,335
1,MG,Clima,84
0,MG,Ausente,83
3,MG,Perdas,76
6,PR,Outros,405
5,PR,Clima,86
7,PR,Perdas,74
4,PR,Ausente,69
10,RJ,Outros,361
8,RJ,Ausente,79


Principal motivo de atrado de cada filial